<a href="https://colab.research.google.com/github/elle-et-noire/lambda-talk/blob/main/notebooks/lambda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Python からクラスも数も消す：手を動かすノートブック

講義スライド「Python からクラスも数も消す（ラムダ計算入門）」の内容を、
Google Colab 上で実際に動かして試すためのノートブックです。

- Colab で開く：[https://colab.research.google.com/github/elle-et-noire/lambda-talk/blob/main/notebooks/lambda.ipynb](https://colab.research.google.com/github/elle-et-noire/lambda-talk/blob/main/notebooks/lambda.ipynb)
- リポジトリ（スライド・参照実装）：[https://github.com/elle-et-noire/lambda-talk](https://github.com/elle-et-noire/lambda-talk)

**§番号＝スライドの章番号** です（§3 ＝ 3 章「チャーチ真偽値」）。
各小見出しの「スライド：…」は、対応するスライドのタイトルです。

## §0 使い方

- 上から順に `Shift + Enter` で実行します。前のセルの定義を後のセルが使うので、**順番が大事** です。
  途中から始めたいときは「ランタイム → 以前のセルを実行」が便利です。
- 「ランタイム → すべてのセルを実行」でも最後までエラーなく通ります。
  わざとエラー（`RecursionError` など）を見せるセルは `try` / `except` で包んであり、エラー名を表示するだけです。
- **変えてみよう**：数や引数を書き換えて、何が起きるか試すセルです。
- **演習**：`MY_OR = None` のような行を自分の定義に書き換えて実行すると、答え合わせが出ます。
- **解答** セルは Colab では折りたたまれています（タイトルだけ表示）。自分で考えてから、ダブルクリックで開いてください。
  解答セルは後の節で使う正しい定義も読み込むので、**開かなくても実行だけはしてください**（すべて実行なら自動で実行されます）。
- 外部パッケージは使いません。Python 3.10 以上で動きます。リポジトリの `code/lambda_core.py` は読み込まず、必要な定義はすべてこのノートブックの中に書いてあります。

### ルール（スライド：「ルール」「例外：観測用関数＝窓」）

| 使ってよい | 使わない |
|---|---|
| 1 引数の `lambda` | 数値リテラル `0`、`1`、…… |
| 関数適用 `f(x)` | `True`、`False`、`if` |
| 名前付け（長い式の略記） | `class`、`+`、`==` などの演算子、`def` による再帰 |

例外は **窓**（`to_bool`、`to_int`、`to_list`、`to_str` など `to_` で始まる観測用の関数）です。
窓は純粋な世界の外側から結果を覗くだけで、計算には使いません。

このノートブックでは、答え合わせ用の `check` や、実験用に数を作る `to_church` などの
**道具**（純粋な世界の外のふつうの Python）も使います。下のセルで準備します。

In [ ]:
import sys
import time

print("Python", sys.version.split()[0])


def check(candidate, tests):
    """演習の答え合わせ用の道具（純粋な世界の外側）。

    tests は (説明, 候補を受け取って結果を返す関数, 期待値) のリスト。
    """
    if candidate is None:
        print("まだ None のままです。定義を書いてから実行しよう。")
        return
    ok = 0
    for label, run, expected in tests:
        try:
            got = run(candidate)
        except Exception as error:  # RecursionError なども表示だけする
            got = type(error).__name__
        ok += got == expected
        mark = "OK" if got == expected else "NG"
        print(f"{mark}  {label} -> {got!r}（期待値 {expected!r}）")
    print(f"{ok}/{len(tests)} 件正解")

## §1 導入：λ記法と Python の lambda（スライド 1 章）

スライド：「λ記法と Python の lambda」

| λ記法 | Python |
|---|---|
| $\lambda x.\,x$ | `lambda x: x` |
| $\lambda x.\,\lambda y.\,x$ | `lambda x: lambda y: x` |
| $\lambda f.\,\lambda x.\,f\,x$ | `lambda f: lambda x: f(x)` |
| $(\lambda x.\,x)\;(\lambda y.\,y)$ | `(lambda x: x)(lambda y: y)` |

$\lambda$ → `lambda`、ドット → コロン、並べて書く → 括弧で呼ぶ。

In [ ]:
# lambda で作った値は「関数」としか表示されない
print(lambda x: x)
print((lambda x: x)(lambda y: y))

## §2 文法と評価（スライド 2 章）

項は 3 種類だけ：変数 $x$、抽象 $\lambda x.\,M$（関数を作る）、適用 $M\,N$（関数を呼ぶ）。

### 恒等関数 `ID`（スライド：「Python に写すと」）

自由な名前 a, b の代わりに、外側の値として文字列 `"a"`、`"b"` を渡して観察します。

In [ ]:
ID = lambda x: x

print(repr(ID("a")))
print(repr(ID(ID)("b")))

### 自由変数と束縛変数・α変換（スライド：「自由変数と束縛変数」「α変換：束縛変数の名前は自由」）

内側の `lambda x: y` だけを見ると、`y` は外側から借りてくる自由変数です。
束縛変数の名前は付け替えても同じふるまいです（α変換）。

In [ ]:
print(repr((lambda y: lambda x: y)("外")("a")))

# α変換：引数名が違うだけの関数は同じふるまい
print(repr((lambda x: x)("a")), repr((lambda y: y)("a")))

### β簡約とカリー化（スライド：「β簡約を 1 ステップずつ」「カリー化：`f(a)(b)` の読み方」）

$(\lambda x.\,\lambda y.\,x)\;a\;b \to_\beta (\lambda y.\,a)\;b \to_\beta a$。
`f(a)` は「残りの引数を待つ関数」を返します。

In [ ]:
print(repr((lambda x: lambda y: x)("a")("b")))

half = (lambda x: lambda y: x)("a")   # まだ y を待っている関数
print(half)
print(repr(half("b")))

In [ ]:
# 変えてみよう：2 つ目を返す関数にすると？ "b" を別の値にすると？
print(repr((lambda x: lambda y: y)("a")("b")))

### 演習：手で β簡約する（スライド：「手で β簡約する」）

紙の上で 1 ステップずつ簡約してから、下の解答セルで Python でも確かめましょう。

1. $(\lambda x.\,x\;x)\;(\lambda y.\,y)$
2. $(\lambda x.\,\lambda y.\,y\;x)\;a\;(\lambda z.\,z)$

In [ ]:
#@title 解答：手で β簡約する（自分で考えてから開く）

# 1. (λx. x x)(λy. y) →β (λy. y)(λy. y) →β λy. y   …恒等関数になる
r1 = (lambda x: x(x))(lambda y: y)
print(repr(r1("a")))   # 恒等関数なので "a" がそのまま返る

# 2. (λx. λy. y x) a (λz. z) →β (λy. y a)(λz. z) →β (λz. z) a →β a
print(repr((lambda x: lambda y: y(x))("a")(ID)))

### 発展：η変換と「包むと評価が遅れる」（スライド：「η変換 (eta conversion)」）

`lambda v: f(v)` と `f` は呼べば同じ結果。ただし `f` を **計算するタイミング** が違います（6 章の Z の伏線）。
下の `make` は「呼ばれたことが分かるように print する」観察用の道具です。

In [ ]:
make = lambda _: print("  （ここで f を計算した）") or ID   # 観察用の道具

print("f1 を作る：")
f1 = make(None)                  # すぐ計算される
print("f2 を作る：")
f2 = lambda v: make(None)(v)     # η展開：まだ計算されない
print("f2 を呼ぶ：")
print(repr(f2("a")))             # 呼んだときに初めて計算される

### 発展：Python は正格評価（スライド：「評価戦略：どの簡約基から簡約するか」「Python は正格評価（値呼び）」）

関数を呼ぶ前に、使わない引数でも先に計算します。ただし **lambda の中身は呼ばれるまで計算しない**。

In [ ]:
print(repr((lambda x: "a")(print("引数を先に評価"))))

# 変えてみよう：lambda _: で包むと、使われない引数の中身は計算されない
print(repr((lambda x: "a")(lambda _: print("これは表示されない"))))

## §3 チャーチ真偽値（スライド 3 章）

### `TRUE` と `FALSE`（スライド：「真偽値は「2 択から選ぶ関数」」「`TRUE` と `FALSE`」）

真：2 つ受け取り 1 つ目を返す。偽：2 つ受け取り 2 つ目を返す。

In [ ]:
TRUE = lambda t: lambda f: t
FALSE = lambda t: lambda f: f

print(repr(TRUE("then 側")("else 側")))
print(repr(FALSE("then 側")("else 側")))
print(TRUE)   # そのまま表示しても中身は見えない

### 窓その1：`to_bool`（スライド：「窓その1：`to_bool`」）

In [ ]:
to_bool = lambda b: b(True)(False)

print(to_bool(TRUE), to_bool(FALSE))

### `IF`、`NOT`、`AND`（スライド：「`IF`：実は何もしない」「`NOT` と `AND`：選ばせ方を工夫する」）

In [ ]:
IF = lambda b: lambda t: lambda f: b(t)(f)
NOT = lambda b: b(FALSE)(TRUE)
AND = lambda a: lambda b: a(b)(FALSE)

print(repr(IF(TRUE)("a")("b")), repr(IF(FALSE)("a")("b")))
print(to_bool(NOT(TRUE)))
print(to_bool(AND(TRUE)(TRUE)), to_bool(AND(TRUE)(FALSE)))

In [ ]:
# 変えてみよう：真理値表を全部出す。AND を NOT や自分で作った演算に替えてみよう
for a in [TRUE, FALSE]:
    for b in [TRUE, FALSE]:
        print(to_bool(a), to_bool(b), "->", to_bool(AND(a)(b)))

### 演習：`OR` を書く（スライド：「`OR` を書く」）

`MY_OR = lambda a: lambda b: ＿＿＿` を完成させよう。
ヒント：AND と同じく「a に 2 択を選ばせる」。2 つの候補は何か。

In [ ]:
MY_OR = None   # ← lambda a: lambda b: ... に書き換える

check(MY_OR, [
    ("OR(TRUE)(TRUE)", lambda OR: to_bool(OR(TRUE)(TRUE)), True),
    ("OR(TRUE)(FALSE)", lambda OR: to_bool(OR(TRUE)(FALSE)), True),
    ("OR(FALSE)(TRUE)", lambda OR: to_bool(OR(FALSE)(TRUE)), True),
    ("OR(FALSE)(FALSE)", lambda OR: to_bool(OR(FALSE)(FALSE)), False),
])

In [ ]:
#@title 解答：OR（スライド「`OR` の解答」）（自分で考えてから開く）

OR = lambda a: lambda b: a(TRUE)(b)

print(to_bool(OR(FALSE)(TRUE)), to_bool(OR(FALSE)(FALSE)))

### 落とし穴と対策：サンク（スライド：「落とし穴：`IF` が両方の枝を計算してしまう」「対策：サンク `lambda _: ...` で包む」）

Python は関数を呼ぶ **前に** 引数をすべて計算します（正格評価）。
else 側に止まらない式 `(lambda x: x(x))(lambda x: x(x))` を置くと、then が選ばれるはずでも失敗します。
（エラーを見せるセルなので `try` で包んであります。）

In [ ]:
try:
    to_bool(IF(TRUE)(TRUE)(
        (lambda x: x(x))(lambda x: x(x))))
except RecursionError as error:
    print("RecursionError:", error)

In [ ]:
# 対策：両方の枝を lambda _: で包み、選んだ包みに (ID) を渡して開ける
print(to_bool(IF(TRUE)(lambda _: TRUE)(
    lambda _: (lambda x: x(x))(lambda x: x(x)))(ID)))

In [ ]:
# 変えてみよう：最後の (ID) を付け忘れると？ 包みを開けないので「関数」が返るだけ
print(IF(TRUE)(lambda _: TRUE)(lambda _: FALSE))
print(to_bool(IF(TRUE)(lambda _: TRUE)(lambda _: FALSE)(ID)))
# 上のセルの IF(TRUE) を IF(FALSE) に変えると、止まらない側が選ばれて RecursionError

### 発展・演習：`XOR` を書く（スライド：「`XOR` を書く」）

ヒント：a に 2 択を選ばせる。使ってよいのは `NOT`。

In [ ]:
MY_XOR = None   # ← lambda a: lambda b: ... に書き換える

check(MY_XOR, [
    ("XOR(TRUE)(TRUE)", lambda XOR: to_bool(XOR(TRUE)(TRUE)), False),
    ("XOR(TRUE)(FALSE)", lambda XOR: to_bool(XOR(TRUE)(FALSE)), True),
    ("XOR(FALSE)(TRUE)", lambda XOR: to_bool(XOR(FALSE)(TRUE)), True),
    ("XOR(FALSE)(FALSE)", lambda XOR: to_bool(XOR(FALSE)(FALSE)), False),
])

In [ ]:
#@title 解答：XOR（スライド「`XOR` の解答」）（自分で考えてから開く）

XOR = lambda a: lambda b: a(NOT(b))(b)

print(to_bool(XOR(TRUE)(FALSE)), to_bool(XOR(TRUE)(TRUE)))
print(to_bool(XOR(FALSE)(TRUE)), to_bool(XOR(FALSE)(FALSE)))

## §4 チャーチ数（スライド 4 章）

数 n ＝「f を n 回適用する関数」：$\overline{n} = \lambda f.\,\lambda x.\,f^n\,x$

### `ZERO`、`ONE` と窓その2 `to_int`（スライド：「`ZERO` と `ONE`」「窓その2：`to_int`」）

In [ ]:
ZERO = lambda f: lambda x: x
ONE = lambda f: lambda x: f(x)

to_int = lambda n: n(lambda x: x + 1)(0)

print(to_int(ZERO), to_int(ONE))

### `SUCC` と `TEN` まで（スライド：「`SUCC`：もう 1 回 f を適用する」「`TEN` まで名前を付けておく」）

In [ ]:
SUCC = lambda n: lambda f: lambda x: f(n(f)(x))
TWO = SUCC(ONE)
THREE = SUCC(TWO)
FOUR = SUCC(THREE)
FIVE = SUCC(FOUR)
SIX = SUCC(FIVE)
SEVEN = SUCC(SIX)
EIGHT = SUCC(SEVEN)
NINE = SUCC(EIGHT)
TEN = SUCC(NINE)

print(to_int(THREE), to_int(TEN))

In [ ]:
# 変えてみよう：窓を取り替える。f と x に文字列の関数を渡すと「何回適用したか」が見える
print(THREE(lambda s: "f(" + s + ")")("x"))
print(ZERO(lambda s: "f(" + s + ")")("x"))

#### 実験用の道具 `to_church`（スライドには出てきません）

Python の整数からチャーチ数を作る、窓の逆向きの道具です。ふつうの Python（`for`）で
`ZERO` に `SUCC` を k 回かぶせるだけ。純粋な世界の定義には使わず、実験の入力を作るためだけに使います。

In [ ]:
def to_church(k):
    n = ZERO
    for _ in range(k):
        n = SUCC(n)
    return n


print(to_int(to_church(42)))

### 足し算と掛け算（スライド：「`ADD`：`SUCC` を m 回」「掛け算：まずは足し算の繰り返しで」「`MUL`：関数合成 (composition) だけ」）

In [ ]:
ADD = lambda m: lambda n: m(SUCC)(n)
MUL_BY_ADD = lambda m: lambda n: m(ADD(n))(ZERO)   # 発展スライドの説明用
MUL = lambda m: lambda n: lambda f: m(n(f))

print(to_int(ADD(TWO)(THREE)))
print(to_int(MUL_BY_ADD(TWO)(THREE)))
print(to_int(MUL(THREE)(FOUR)))

In [ ]:
# 変えてみよう：MUL(TWO)(THREE) に文字列の窓を当てると、f が 6 回かぶさっている
print(MUL(TWO)(THREE)(lambda s: "f(" + s + ")")("x"))

### 演習：`POW` を書く（スライド：「`POW` を書く（`EXP` とも）」）

`MY_POW = lambda m: lambda n: ＿＿＿`（$m^n$）。`MUL` より短く書けます。

ヒント：数 m 自身も「関数 f を受け取り、f の m 回合成を返す関数」。それを n 回繰り返すと？

In [ ]:
MY_POW = None   # ← lambda m: lambda n: ... に書き換える

check(MY_POW, [
    ("POW(TWO)(THREE)", lambda POW: to_int(POW(TWO)(THREE)), 8),
    ("POW(THREE)(TWO)", lambda POW: to_int(POW(THREE)(TWO)), 9),
    ("POW(FIVE)(ZERO)", lambda POW: to_int(POW(FIVE)(ZERO)), 1),
])

In [ ]:
#@title 解答：POW（スライド「`POW` の解答：`n` に `m` を渡すだけ」）（自分で考えてから開く）

POW = lambda m: lambda n: n(m)

print(to_int(POW(TWO)(THREE)), to_int(POW(TWO)(TEN)))

### `IS_ZERO`（スライド：「`IS_ZERO`：1 回でも適用されたら偽」）

In [ ]:
IS_ZERO = lambda n: n(lambda _: FALSE)(TRUE)

print(to_bool(IS_ZERO(ZERO)), to_bool(IS_ZERO(THREE)))

### 発展・演習：`IS_EVEN` を書く（スライド：「`IS_EVEN` を書く」）

ヒント：0 は偶数。1 増えるたびに偶奇は反転する。反転する関数は 3 章にあった。

In [ ]:
MY_IS_EVEN = None   # ← lambda n: ... に書き換える

check(MY_IS_EVEN, [
    ("IS_EVEN(ZERO)", lambda IS_EVEN: to_bool(IS_EVEN(ZERO)), True),
    ("IS_EVEN(FOUR)", lambda IS_EVEN: to_bool(IS_EVEN(FOUR)), True),
    ("IS_EVEN(SEVEN)", lambda IS_EVEN: to_bool(IS_EVEN(SEVEN)), False),
])

In [ ]:
#@title 解答：IS_EVEN（スライド「`IS_EVEN` の解答」）（自分で考えてから開く）

IS_EVEN = lambda n: n(NOT)(TRUE)

print(to_bool(IS_EVEN(FOUR)), to_bool(IS_EVEN(SEVEN)))

## §5 ペアと引き算（スライド 5 章）

### ペア（スライド：「ペア：2 つを覚えて、選び手に渡す」）

In [ ]:
PAIR = lambda a: lambda b: lambda s: s(a)(b)
FIRST = lambda p: p(TRUE)
SECOND = lambda p: p(FALSE)

print(to_int(FIRST(PAIR(ONE)(TWO))), to_int(SECOND(PAIR(ONE)(TWO))))

In [ ]:
# 変えてみよう：選び手 s に TRUE / FALSE 以外を渡すと？
p = PAIR("左")("右")
print(repr(p(TRUE)), repr(p(FALSE)))
print(repr(p(lambda a: lambda b: b + a)))   # 観察用：2 つを逆順につなぐ

### Kleene のトリック：`SHIFT` と `PRED`（スライド：「引き算はなぜ難しいか」「Kleene のトリック：1 つ遅れて追いかける」「`SHIFT` と `PRED`」）

ペア（前, 今）を持ち歩き、1 回ごとに今を前へずらし、今は +1：$(a, b) \mapsto (b, b+1)$

In [ ]:
SHIFT = lambda p: PAIR(SECOND(p))(SUCC(SECOND(p)))
PRED = lambda n: FIRST(n(SHIFT)(PAIR(ZERO)(ZERO)))

print(to_int(PRED(FIVE)), to_int(PRED(ZERO)))

In [ ]:
# スライドの表を再現する。to_pair はペアを覗く窓（このノートブック用）
to_pair = lambda p: (to_int(FIRST(p)), to_int(SECOND(p)))

p = PAIR(ZERO)(ZERO)
for k in range(6):          # 変えてみよう：回数を増やす
    print(k, "回:", to_pair(p))
    p = SHIFT(p)

### 発展・演習：`SUB` を書く（スライド：「`SUB` を書く」）

`MY_SUB(m)(n)` が m − n（m < n なら 0）になるように。
ヒント：数 n は「何かを n 回適用する」関数。m に何を n 回適用すればよい？ 4 章の ADD も思い出す。

In [ ]:
MY_SUB = None   # ← lambda m: lambda n: ... に書き換える

check(MY_SUB, [
    ("SUB(SEVEN)(THREE)", lambda SUB: to_int(SUB(SEVEN)(THREE)), 4),
    ("SUB(TWO)(FIVE)", lambda SUB: to_int(SUB(TWO)(FIVE)), 0),
    ("SUB(FIVE)(ZERO)", lambda SUB: to_int(SUB(FIVE)(ZERO)), 5),
])

In [ ]:
#@title 解答：SUB（スライド「`SUB` の解答」。後の LEQ で使うので必ず実行）（自分で考えてから開く）

SUB = lambda m: lambda n: n(PRED)(m)

print(to_int(SUB(SEVEN)(THREE)), to_int(SUB(TWO)(FIVE)))

### 発展：大小比較 `LEQ` と `EQ`（スライド：「大小比較：`LEQ` と `EQ`」）

In [ ]:
LEQ = lambda m: lambda n: IS_ZERO(SUB(m)(n))
EQ = lambda m: lambda n: AND(LEQ(m)(n))(LEQ(n)(m))

print(to_bool(LEQ(THREE)(FIVE)), to_bool(EQ(FOUR)(FIVE)))

In [ ]:
# 変えてみよう：遅さを測る。PRED は 1 回ごとに SHIFT を n 回、SUB はその PRED を n 回
for k in [10, 20, 40]:
    n = to_church(k)
    start = time.perf_counter()
    to_bool(EQ(n)(n))
    print(k, f"{time.perf_counter() - start:.4f} 秒")

## §6 再帰と Y コンビネータ（スライド 6 章）

### 自分の名前は呼べない（スライド：「階乗を書きたい。でも自分の名前は呼べない」）

下の `FACT_BAD` は **ルール違反の比較用** です。右辺に自分の名前がいます。
Python では、呼び出した時点でグローバル変数を探しに行く仕組みのおかげで動いてしまいますが、
ラムダ計算にはその仕組みがありません（名前は長い式の略記にすぎない）。

In [ ]:
FACT_BAD = lambda n: (          # ルール違反：右辺に FACT_BAD 自身
    IF(IS_ZERO(n))
      (lambda _: ONE)
      (lambda _: MUL(n)(FACT_BAD(PRED(n))))
      (ID)
)

print(to_int(FACT_BAD(FIVE)), "← 動くが、Python の名前の仕組みに頼った反則")

### 手がかり：自分を自分に渡す（スライド：「手がかり：自分を自分に渡す」）

In [ ]:
omega = lambda x: x(x)

try:
    omega(omega)
except RecursionError as error:
    print("RecursionError:", error)

### `FACT_GEN`：`self(self)` で再帰する（スライド：「アイデア：自分を引数でもらう」「`FACT_GEN`：`self(self)` で再帰する」「動いた：`FACT_GEN(FACT_GEN)`」）

In [ ]:
FACT_GEN = lambda self: lambda n: (
    IF(IS_ZERO(n))
      (lambda _: ONE)
      (lambda _: MUL(n)(self(self)(PRED(n))))
      (ID)
)

print(to_int(FACT_GEN(FACT_GEN)(THREE)), to_int(FACT_GEN(FACT_GEN)(FIVE)))

In [ ]:
# 変えてみよう：self(self) を self に変えると？
# self は「まだ self を待っている関数」なので、n を渡すと形が合わない
FACT_GEN_WRONG = lambda self: lambda n: (
    IF(IS_ZERO(n))
      (lambda _: ONE)
      (lambda _: MUL(n)(self(PRED(n))))
      (ID)
)
try:
    print(to_int(FACT_GEN_WRONG(FACT_GEN_WRONG)(THREE)))
except Exception as error:
    print(type(error).__name__ + ":", error)

### `FACT_BODY`：ふつうの再帰の形（スライド：「`self(self)` を外へ追い出す」「`FACT_BODY`：ふつうの再帰の形」）

再帰呼び出しの相手 `fact` を引数でもらいます。本物の階乗を `fact` に渡せば階乗になります。

In [ ]:
FACT_BODY = lambda fact: lambda n: (
    IF(IS_ZERO(n))
      (lambda _: ONE)
      (lambda _: MUL(n)(fact(PRED(n))))
      (ID)
)

print(to_int(FACT_BODY(FACT_GEN(FACT_GEN))(FIVE)))

In [ ]:
# 変えてみよう：偽物の fact（いつも 0 を返す）を渡すと？ n = 0 のときだけは正しい
fake = lambda n: ZERO
print(to_int(FACT_BODY(fake)(ZERO)), to_int(FACT_BODY(fake)(THREE)))

### Y コンビネータを Python で使うと……（スライド：「組み立て直すと Y コンビネータ」「Y F は F の不動点」「Python で Y を使うと……」「なぜ？ Python は引数を先に計算する」）

$Y = \lambda f.\,(\lambda x.\,f\,(x\,x))\,(\lambda x.\,f\,(x\,x))$。理屈は正しいのに、
Python では数を渡す前、`Y(FACT_BODY)` を **作るだけで** 落ちます。

In [ ]:
Y = lambda f: (lambda x: f(x(x)))(lambda x: f(x(x)))

try:
    Y(FACT_BODY)
except RecursionError as error:
    print("RecursionError:", error)

### 包んで遅らせる：η展開と Z コンビネータ（スライド：「包んで遅らせる：η展開」「Z コンビネータ：Y の x x を包む」「`FACT = Z(FACT_BODY)`」）

In [ ]:
# 包むだけなら落ちない（中身は呼ばれるまで計算しない）
print(lambda v: omega(omega)(v))

In [ ]:
Z = lambda f: (
    (lambda x: f(lambda v: x(x)(v)))
    (lambda x: f(lambda v: x(x)(v)))
)
FACT = Z(FACT_BODY)

print(to_int(FACT(ZERO)), to_int(FACT(THREE)))
print(to_int(FACT(FIVE)))

In [ ]:
# 変えてみよう：IF の枝をサンクで包まないと、n = 0 でも else 側を計算して止まらない
FACT_BODY_STRICT = lambda fact: lambda n: (
    IF(IS_ZERO(n))(ONE)(MUL(n)(fact(PRED(n)))))
try:
    print(to_int(Z(FACT_BODY_STRICT)(THREE)))
except RecursionError as error:
    print("RecursionError:", error)

### 演習：Z で `SUM_TO_N` を書く（スライド：「Z で `SUM_TO_N` を書く」）

0 + 1 + … + n を計算する関数。ヒント：FACT_BODY をまねる。n = 0 なら ZERO、そうでなければ n + sum_to_n(n − 1)。
`...` の 2 か所を埋めて、`MY_SUM_TO_N = None` の行を消してください。

In [ ]:
MY_SUM_TO_N = None   # ← 下のひな形を使うなら、この行を消す
# MY_SUM_TO_N = Z(lambda sum_to_n: lambda n: (
#     IF(IS_ZERO(n))
#       (lambda _: ...)
#       (lambda _: ...)
#       (ID)
# ))

check(MY_SUM_TO_N, [
    ("SUM_TO_N(ZERO)", lambda S: to_int(S(ZERO)), 0),
    ("SUM_TO_N(FOUR)", lambda S: to_int(S(FOUR)), 10),
    ("SUM_TO_N(TEN)", lambda S: to_int(S(TEN)), 55),
])

In [ ]:
#@title 解答：SUM_TO_N（スライド「`SUM_TO_N` の解答」）（自分で考えてから開く）

SUM_TO_N = Z(lambda sum_to_n: lambda n: (
    IF(IS_ZERO(n))
      (lambda _: ZERO)
      (lambda _: ADD(n)(sum_to_n(PRED(n))))
      (ID)
))

print(to_int(SUM_TO_N(FOUR)), to_int(SUM_TO_N(TEN)))

### 発展：`FIB`（スライド：「`FIB`：再帰呼び出しが 2 つ」）

In [ ]:
FIB = Z(lambda fib: lambda n: (
    IF(IS_ZERO(PRED(n)))
      (lambda _: n)
      (lambda _: ADD(fib(PRED(n)))
                       (fib(PRED(PRED(n)))))
      (ID)
))

print(to_int(FIB(TEN)))

In [ ]:
# 変えてみよう：いろいろな n で。FACT は 8、FIB は 12 くらいまでが快適
print([to_int(FACT(to_church(k))) for k in range(8)])
print([to_int(FIB(to_church(k))) for k in range(13)])

## §7 リストと FizzBuzz（スライド 7 章・発展）

5 章発展の `SUB`・`LEQ` を使います（このノートブックでは定義済み）。

### リストの表現（スライド：「リストの表現：ペアの入れ子」「`NIL`、`CONS`、`IS_NIL`、`HEAD`、`TAIL`」）

1 つ目は「空か？」の旗。中身は 2 つ目のペアに入れます。

In [ ]:
NIL = PAIR(TRUE)(TRUE)
CONS = lambda h: lambda t: PAIR(FALSE)(PAIR(h)(t))
IS_NIL = FIRST
HEAD = lambda l: FIRST(SECOND(l))
TAIL = lambda l: SECOND(SECOND(l))

print(to_int(HEAD(CONS(ONE)(NIL))))
print(to_bool(IS_NIL(NIL)), to_bool(IS_NIL(CONS(ONE)(NIL))))

### 窓その3：`to_list`（スライド：「窓その3：`to_list`」）

`def` と `while` を使いますが、観測用の窓なので可。

In [ ]:
def to_list(l, conv=lambda x: x):
    result = []
    while not to_bool(IS_NIL(l)):
        result.append(conv(HEAD(l)))
        l = TAIL(l)
    return result


print(to_list(CONS(ONE)(CONS(TWO)(NIL)), to_int))

### `RANGE`・`FOLD`・`MAP`（スライド：「`RANGE`：m から n までのリスト」「畳み込み (fold) のアイデア」「`FOLD`」「`MAP`：`FOLD` で書く」）

In [ ]:
RANGE = Z(lambda range_: lambda m: lambda n: (
    IF(LEQ(m)(n))
      (lambda _: CONS(m)(range_(SUCC(m))(n)))
      (lambda _: NIL)
      (ID)
))

print(to_list(RANGE(ONE)(FIVE), to_int))

In [ ]:
FOLD = Z(lambda fold: lambda f: lambda z: lambda l: (
    IF(IS_NIL(l))
      (lambda _: z)
      (lambda _: f(HEAD(l))(fold(f)(z)(TAIL(l))))
      (ID)
))

print(to_list(FOLD(CONS)(NIL)(RANGE(ONE)(THREE)), to_int))
print(to_int(FOLD(lambda _: SUCC)(ZERO)(RANGE(ONE)(FIVE))))   # 長さ

In [ ]:
# 変えてみよう：FOLD の f と z に文字列の関数を渡すと、「CONS → f、NIL → z」の置き換えが見える
show = lambda x: lambda acc: "f(" + str(to_int(x)) + ")(" + acc + ")"
print(FOLD(show)("z")(RANGE(ONE)(THREE)))

In [ ]:
MAP = lambda f: lambda l: (
    FOLD(lambda x: lambda acc: CONS(f(x))(acc))(NIL)(l)
)

print(to_list(MAP(SUCC)(RANGE(ONE)(THREE)), to_int))

### 演習：`SUM` を `FOLD` で書く（スライド：「`SUM` を `FOLD` で書く」）

使ってよい：`FOLD`、`ADD`、`ZERO`。ヒント：CONS を何に、NIL を何に置き換える？

In [ ]:
MY_SUM = None   # ← ここに書く（1 行で書ける）

check(MY_SUM, [
    ("SUM(RANGE(ONE)(FOUR))", lambda SUM: to_int(SUM(RANGE(ONE)(FOUR))), 10),
    ("SUM(NIL)", lambda SUM: to_int(SUM(NIL)), 0),
])

In [ ]:
#@title 解答：SUM（スライド「`SUM` の解答：1 行」）（自分で考えてから開く）

SUM = FOLD(ADD)(ZERO)

print(to_int(SUM(RANGE(ONE)(FOUR))))

### FizzBuzz の部品：`MOD`、`DIV`、`PUSH`、`TO_DIGITS`（スライド：「FizzBuzz を関数だけで」「`MOD`：引けるだけ引く」「`DIV`：引けた回数を数える」「`PUSH` と `TO_DIGITS`：数を数字の並びに」）

前提：除数 n は 1 以上。n = ZERO は未定義で、この実装では止まりません。

In [ ]:
MOD = Z(lambda mod: lambda m: lambda n: (
    IF(LEQ(n)(m))
      (lambda _: mod(SUB(m)(n))(n))
      (lambda _: m)
      (ID)
))
DIV = Z(lambda div: lambda m: lambda n: (
    IF(LEQ(n)(m))
      (lambda _: SUCC(div(SUB(m)(n))(n)))
      (lambda _: ZERO)
      (ID)
))

print(to_int(MOD(TEN)(THREE)), to_int(DIV(TEN)(THREE)))

In [ ]:
PUSH = lambda l: lambda x: FOLD(CONS)(CONS(x)(NIL))(l)
TO_DIGITS = Z(lambda digits: lambda n: (
    IF(LEQ(n)(NINE))
      (lambda _: CONS(n)(NIL))
      (lambda _: PUSH(digits(DIV(n)(TEN)))
                       (MOD(n)(TEN)))
      (ID)
))

print(to_list(TO_DIGITS(POW(TEN)(TWO)), to_int))

#### チャレンジ：`REVERSE`（スライドには出てきません）

リストを逆順にする `REVERSE` を `FOLD` と `PUSH` で書いてみよう。
ヒント：右畳み込みは右端から処理する。「残りを逆順にしたもの」の末尾に先頭を足せばよい。

In [ ]:
MY_REVERSE = None   # ← ここに書く

check(MY_REVERSE, [
    ("REVERSE(RANGE(ONE)(FOUR))",
     lambda R: to_list(R(RANGE(ONE)(FOUR)), to_int), [4, 3, 2, 1]),
    ("REVERSE(NIL)", lambda R: to_list(R(NIL), to_int), []),
])

In [ ]:
#@title 解答：REVERSE（自分で考えてから開く）

REVERSE = FOLD(lambda x: lambda acc: PUSH(acc)(x))(NIL)

print(to_list(REVERSE(RANGE(ONE)(FOUR)), to_int))

### 文字と窓その4 `to_str`（スライド：「文字の表現：0〜14 の数で」「窓その4：`to_str`」）

| 数 | 0〜9 | 10 | 11 | 12 | 13 | 14 |
|---|---|---|---|---|---|---|
| 文字 | 0〜9 | B | F | i | u | z |

In [ ]:
ELEVEN = SUCC(TEN)
TWELVE = SUCC(ELEVEN)
THIRTEEN = SUCC(TWELVE)
FOURTEEN = SUCC(THIRTEEN)

to_str = lambda l: "".join(
    to_list(l, lambda c: "0123456789BFiuz"[to_int(c)])
)

print(repr(to_str(TO_DIGITS(FOURTEEN))))

### FizzBuzz を動かす（スライド：「`FIZZ`、`BUZZ`、`FIZZBUZZ_WORD`」「`FIZZBUZZ_ITEM`：1 つの数を文字列に」「`FIZZBUZZ`：動かす」）

In [ ]:
FIZZ = CONS(ELEVEN)(
    CONS(TWELVE)(CONS(FOURTEEN)(CONS(FOURTEEN)(NIL)))
)
BUZZ = CONS(TEN)(
    CONS(THIRTEEN)(CONS(FOURTEEN)(CONS(FOURTEEN)(NIL)))
)
FIZZBUZZ_WORD = FOLD(CONS)(BUZZ)(FIZZ)

print(to_str(FIZZ), to_str(BUZZ), to_str(FIZZBUZZ_WORD))

In [ ]:
FIZZBUZZ_ITEM = lambda n: (
    IF(IS_ZERO(MOD(n)(THREE)))
      (lambda _: IF(IS_ZERO(MOD(n)(FIVE)))
          (lambda _: FIZZBUZZ_WORD)
          (lambda _: FIZZ)
          (ID))
      (lambda _: IF(IS_ZERO(MOD(n)(FIVE)))
          (lambda _: BUZZ)
          (lambda _: TO_DIGITS(n))
          (ID))
      (ID))
FIZZBUZZ = lambda n: MAP(FIZZBUZZ_ITEM)(RANGE(ONE)(n))
FIFTEEN = MUL(THREE)(FIVE)

print(to_list(FIZZBUZZ(FIFTEEN), to_str))

In [ ]:
# 変えてみよう：範囲を広げる。再帰上限 1000 のままでも 200 程度までは動く（少し待つ）
start = time.perf_counter()
print(to_list(FIZZBUZZ(to_church(30)), to_str))
print(f"{time.perf_counter() - start:.3f} 秒")

## §8 SKI コンビネータ（スライド 8 章・発展）

### `S`、`K`、`I`（スライド：「lambda すら消す：`S`、`K`、`I`」「`I = S K K` を確かめる」）

$S = \lambda x.\,\lambda y.\,\lambda z.\,x\,z\,(y\,z)$、$K = \lambda x.\,\lambda y.\,x$、$I = S\,K\,K$

In [ ]:
S = lambda x: lambda y: lambda z: x(z)(y(z))
K = lambda x: lambda y: x
I = S(K)(K)

print(to_int(I(THREE)), to_bool(I(TRUE)))

### 演習：`K` と `K I` は何に見える？（スライド：「`K` と `K I` は何に見える？」）

$K\,a\,b$ と $K\,I\,a\,b$ を紙の上で簡約して予想してから、解答セルを開こう。

In [ ]:
#@title 解答：K と K I（スライド「解答：`TRUE` と `FALSE`」）（自分で考えてから開く）

print(repr(K("a")("b")), repr(K(I)("a")("b")))
print(to_bool(K), to_bool(K(I)))   # K は TRUE、K I は FALSE のふるまい

### ブラケット抽象の例（スライド：「ブラケット抽象 (bracket abstraction)」「例：$\lambda x.\,x\,x$ → $S\,I\,I$」「例：$\lambda x.\,\mathrm{SUCC}\,x$ → $S\,(K\,\mathrm{SUCC})\,I$」）

$\mathcal{T}[\lambda x.\,x] = I$、$\mathcal{T}[\lambda x.\,E] = K\,\mathcal{T}[E]$（x が E に自由に出ない）、
$\mathcal{T}[\lambda x.\,E_1\,E_2] = S\,\mathcal{T}[\lambda x.\,E_1]\,\mathcal{T}[\lambda x.\,E_2]$

In [ ]:
print(to_int(S(I)(I)(ID)(THREE)))   # S I I は ω と同じふるまい
print(to_int(S(K(SUCC))(I)(TWO)))   # λx. SUCC x

try:
    S(I)(I)(S(I)(I))                # Ω と同じく止まらない
except RecursionError as error:
    print("RecursionError:", error)

#### チャレンジ：ブラケット抽象を手でやる（スライドには出てきません）

1. $\lambda x.\,\mathrm{NOT}\,x$ を S・K・I（と NOT）だけの式にして `MY_T1` に書こう。
2. もう少し難しい：$\lambda x.\,\lambda y.\,y\,x$（2 章の演習 2 の関数）を S・K・I だけにして `MY_T2` に。内側の λ から消す。

In [ ]:
MY_T1 = None   # ← S, K, I, NOT と適用だけで
MY_T2 = None   # ← S, K, I と適用だけで

check(MY_T1, [
    ("T1(TRUE)", lambda t: to_bool(t(TRUE)), False),
    ("T1(FALSE)", lambda t: to_bool(t(FALSE)), True),
])
check(MY_T2, [
    ('T2("a")(ID)', lambda t: t("a")(ID), "a"),
    ("T2(THREE)(SUCC)", lambda t: to_int(t(THREE)(SUCC)), 4),
])

In [ ]:
#@title 解答：ブラケット抽象（自分で考えてから開く）

# T[λx. NOT x] = S T[λx. NOT] T[λx. x] = S (K NOT) I
T1 = S(K(NOT))(I)
# T[λy. y x] = S I (K x)  →  T[λx. S I (K x)] = S (K (S I)) (S (K K) I)
T2 = S(K(S(I)))(S(K(K))(I))

print(to_bool(T1(TRUE)), repr(T2("a")(ID)), to_int(T2(THREE)(SUCC)))

## §9 まとめと先へ（スライド 9 章）

### 今日作ったもの（スライド：「今日作ったもの」）

冒頭で予告したもの。数値リテラルも `if` も `def` も使わずに：

In [ ]:
print(to_int(FACT(FIVE)))

### 性能の話：遅い・すぐ溢れる（スライド：「性能の話：遅い・すぐ溢れる」）

チャーチ数は一進法。`SUCC` の鎖で作った数は、`to_int` で見るときにスタックが深くなります。
Python の再帰上限（既定 1000）を超えると `RecursionError` です。
Colab（Jupyter）ではカーネル自身もスタックを少し使うので、スライドの実測表より少し手前で溢れることがあります。

In [ ]:
print("再帰上限:", sys.getrecursionlimit())

for k in [100, 500, 2000]:     # 変えてみよう：境界を探す
    try:
        print(k, "->", to_int(to_church(k)))
    except RecursionError:
        print(k, "-> RecursionError")

In [ ]:
# FACT の境界はスタックではなく時間。1 つ増やすごとに大きく遅くなる
for k in range(6, 10):
    start = time.perf_counter()
    value = to_int(FACT(to_church(k)))
    print(k, value, f"{time.perf_counter() - start:.3f} 秒")

`sys.setrecursionlimit` で上限を上げることもできますが、上げすぎると Python 自体が落ちる（カーネルが再起動する）ことがあります。試すなら少しずつ。

### この先へ

- スライドと参照実装：[elle-et-noire/lambda-talk](https://github.com/elle-et-noire/lambda-talk)
  （[code/lambda_core.py](https://github.com/elle-et-noire/lambda-talk/blob/main/code/lambda_core.py) にこのノートブックと同じ定義がまとまっています）
- T. Stuart「Programming with Nothing」：Ruby の lambda（proc）だけで FizzBuzz まで書く講演と記事。7 章はそのオマージュ
- 高橋正子『計算論 計算可能性とラムダ計算』、B. C. Pierce『Types and Programming Languages』